# Non working SPH approach. Paused for now

## 3. Pseudo-Axial Power-Shape SPH Demonstration

The pseudo-axial SPH fit clones selected full-height MGXS regions into axial diffusion zones, then fits zone-specific factors from the independent continuous-energy `kappa-fission` mesh. Each clone starts with the same exported full-height MGXS, but the diffusion mesh sees a distinct material label and therefore a distinct corrected XS row after fitting.

The default run below splits every fuel ring into five axial zones: `[-150,-75]`, `[-75,0]`, `[0,75]`, `[75,125]`, and `[125,150]` cm. The central moderator channel is split on the same fuel-height zones plus `[-200,-150]` cm for its lower extension, but its power-shape SPH factors remain inactive/unit because a kappa-fission mesh does not calibrate non-fissile moderator XS.

The three-zone preset is kept in the notebook for quick comparison: `[-150,0]`, `[0,100]`, and `[100,150]` cm.


In [ ]:
# temporary openmc axial region flux recalculation
AXIAL_SPH_THREE_ZONE_PRESET = (
    ("lower", -150.0, 0.0),
    ("upper_bulk", 0.0, 100.0),
    ("rod_tip", 100.0, 150.0),
)
AXIAL_SPH_FIVE_ZONE_PRESET = (
    ("lower_far", -150.0, -75.0),
    ("lower_mid", -75.0, 0.0),
    ("upper_bulk", 0.0, 75.0),
    ("upper_shoulder", 75.0, 125.0),
    ("rod_tip", 125.0, 150.0),
)
AXIAL_SPH_FUEL_ZONES = AXIAL_SPH_FIVE_ZONE_PRESET
AXIAL_SPH_ZONES = {
    label: AXIAL_SPH_FUEL_ZONES
    for label in DIFFUSION_INPUT.fuel_ring_labels
}
AXIAL_SPH_ZONES["core_central_moderator_channel"] = (
    ("lower_extension", -200.0, -150.0),
    *AXIAL_SPH_FUEL_ZONES,
)
if "_mgxs_payload" not in globals():
    _mgxs_payload = json.loads(
        (MGXS_EXPORT_DIR / "outputs" / "mgxs_constants.json").read_text()
    )


def _axial_sph_flux_digest(selected_labels, selected_cell_ids, z_edges):
    model_bytes = DIFFUSION_INPUT.model_xml_path.read_bytes()
    payload = {
        "model_xml_path": str(DIFFUSION_INPUT.model_xml_path),
        "model_sha256": hashlib.sha256(model_bytes).hexdigest(),
        "group_count": DIFFUSION_INPUT.group_count,
        "energy_group_edges_ev": list(DIFFUSION_INPUT.energy_group_edges_ev),
        "selected_labels": list(selected_labels),
        "selected_cell_ids": [int(cell_id) for cell_id in selected_cell_ids],
        "axial_sph_zones": {
            label: [list(entry) for entry in AXIAL_SPH_ZONES[label]]
            for label in selected_labels
        },
        "z_edges_cm": [float(value) for value in z_edges],
    }
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def _load_or_run_temporary_axial_sph_flux():
    selected_labels = tuple(AXIAL_SPH_ZONES)
    z_edges = np.asarray(
        sorted(
            {
                value
                for entries in AXIAL_SPH_ZONES.values()
                for _, z_min, z_max in entries
                for value in (z_min, z_max)
            }
        ),
        dtype=float,
    )
    run_dir = (
        project_root
        / "openmc"
        / "build"
        / "concentric"
        / "axial_sph_flux"
        / f"group_{GROUP_COUNT}"
    )
    reference_json = run_dir / "axial_sph_flux_reference.json"

    probe_model = openmc.Model.from_model_xml(DIFFUSION_INPUT.model_xml_path)
    cells_by_name = {
        cell.name: cell
        for cell in probe_model.geometry.get_all_cells().values()
        if cell.name
    }
    selected_cells = [
        cells_by_name[DIFFUSION_INPUT.domain_mapping[label]]
        for label in selected_labels
    ]
    digest = _axial_sph_flux_digest(
        selected_labels,
        [cell.id for cell in selected_cells],
        z_edges,
    )
    if reference_json.is_file():
        cached = json.loads(reference_json.read_text())
        if cached.get("digest") == digest:
            print(f"Using cached temporary CE axial SPH flux = {reference_json}")
            return cached["axial_region_flux"]

    model = openmc.Model.from_model_xml(DIFFUSION_INPUT.model_xml_path)
    cells_by_name = {
        cell.name: cell
        for cell in model.geometry.get_all_cells().values()
        if cell.name
    }
    selected_cells = [
        cells_by_name[DIFFUSION_INPUT.domain_mapping[label]]
        for label in selected_labels
    ]
    settings = model.settings
    _mgxs_config = _mgxs_payload["config"]
    settings.run_mode = "eigenvalue"
    settings.particles = int(_mgxs_config["particles"])
    settings.batches = int(_mgxs_config["batches"])
    settings.inactive = int(_mgxs_config["inactive"])
    settings.output = {"summary": False}
    model.settings = settings

    mesh = openmc.CylindricalMesh(
        r_grid=np.asarray([0.0, geometry["core_radius_cm"]], dtype=float),
        phi_grid=np.asarray([0.0, 2.0 * np.pi], dtype=float),
        z_grid=z_edges,
        name="temporary-axial-sph-flux-mesh",
    )
    tally = openmc.Tally(name="temporary-axial-sph-flux")
    tally.filters = [
        openmc.CellFilter(selected_cells),
        openmc.MeshFilter(mesh),
        openmc.EnergyFilter(np.asarray(DIFFUSION_INPUT.energy_group_edges_ev, dtype=float)),
    ]
    tally.scores = ["flux"]
    model.tallies = openmc.Tallies([tally])

    run_dir.mkdir(parents=True, exist_ok=True)
    run_kwargs = {
        "cwd": run_dir,
        "export_model_xml": True,
    }
    if "OPENMC_THREADS" in globals():
        run_kwargs["threads"] = OPENMC_THREADS
    print(f"Running temporary CE axial SPH flux tally in {run_dir} ...")
    statepoint_path = Path(model.run(**run_kwargs)).resolve()

    with openmc.StatePoint(statepoint_path) as statepoint:
        loaded = statepoint.get_tally(name="temporary-axial-sph-flux")
        z_bin_count = len(z_edges) - 1
        group_count = DIFFUSION_INPUT.group_count
        mean = np.asarray(loaded.mean, dtype=float).reshape(
            len(selected_cells), z_bin_count, group_count
        )[..., ::-1]
        std_dev = np.asarray(loaded.std_dev, dtype=float).reshape(
            len(selected_cells), z_bin_count, group_count
        )[..., ::-1]

    edge_index = {float(value): index for index, value in enumerate(z_edges)}
    axial_region_flux = {}
    for cell_index, label in enumerate(selected_labels):
        for zone_name, z_min, z_max in AXIAL_SPH_ZONES[label]:
            start = edge_index[float(z_min)]
            stop = edge_index[float(z_max)]
            clone_label = f"{label}__axial_{zone_name}"
            axial_region_flux[clone_label] = {
                "mean": np.sum(mean[cell_index, start:stop, :], axis=0).tolist(),
                "std_dev": np.sqrt(
                    np.sum(std_dev[cell_index, start:stop, :] ** 2, axis=0)
                ).tolist(),
            }

    payload = {
        "digest": digest,
        "statepoint_path": str(statepoint_path),
        "score": "flux",
        "energy_order": "fast-to-thermal",
        "z_edges_cm": z_edges.tolist(),
        "selected_labels": list(selected_labels),
        "axial_region_flux": axial_region_flux,
    }
    reference_json.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")
    print(f"Wrote temporary CE axial SPH flux = {reference_json}")
    return axial_region_flux


_axial_region_flux_payload = _load_or_run_temporary_axial_sph_flux()
DIFFUSION_INPUT = replace(
    DIFFUSION_INPUT,
    ce_reference=replace(
        DIFFUSION_INPUT.ce_reference,
        axial_region_flux={
            label: ReferenceValues(
                mean=values["mean"],
                std_dev=values["std_dev"],
            )
            for label, values in _axial_region_flux_payload.items()
        },
    ),
)
AXIAL_SPH_INPUT = axialized_diffusion_input(DIFFUSION_INPUT, AXIAL_SPH_ZONES)

In [ ]:
# sph solver config
SPH_SETTINGS = SphSettings(
    damping=0.1,
    max_iterations=50,
    flux_tolerance=1.0e-2,
    stable_iterations=2,
    maximum_relative_std_dev=5.0e-2,
    excluded_region_labels=(),
    qualification_power_rms_tolerance=5.0e-2,
    boundary_condition=CACHE_SETTINGS.boundary_condition,
    solver_max_iterations=CACHE_SETTINGS.max_iter,
    solver_tolerance=CACHE_SETTINGS.tol,
    solver_source_tolerance=CACHE_SETTINGS.source_tol,
    solver_max_inner_iterations=CACHE_SETTINGS.max_inner_iter,
    solver_inner_tolerance=CACHE_SETTINGS.inner_tol,
)
SPH_REFERENCE = build_axial_flux_sph_reference(
    DIFFUSION_INPUT,
    SPH_SETTINGS,
    axial_sph_zones=AXIAL_SPH_ZONES,
)

print("=== CONTINUOUS-ENERGY AXIAL FLUX SPH REFERENCE ===")
print(f"CE k_eff uncertainty       = {reference_std_pcm:.1f} pcm")
print(f"Reference fission prod.    = {SPH_REFERENCE.fission_production:.6e}")
print(f"Axialized diffusion zones  = {len(AXIAL_SPH_INPUT.zones)}")
print(
    f"Active flux bins          = {np.count_nonzero(SPH_REFERENCE.active)} / "
    f"{SPH_REFERENCE.active.size}"
)
print(f"Excluded SPH regions       = {SPH_SETTINGS.excluded_region_labels}")
print("Inactive axial flux rows")
for row, label in enumerate(SPH_REFERENCE.region_labels):
    if not np.any(SPH_REFERENCE.active[row]):
        max_relative_std = 100.0 * np.nanmax(SPH_REFERENCE.relative_std_dev[row])
        print(f"  {label:>58}: max rel std={max_relative_std:7.3f}%")

fig, ax = plt.subplots(figsize=(9.5, 6.5))
image = ax.imshow(
    np.where(SPH_REFERENCE.active, SPH_REFERENCE.flux, np.nan),
    aspect='auto',
    cmap='viridis',
)
ax.set(
    xticks=np.arange(DIFFUSION_INPUT.group_count),
    xticklabels=np.arange(1, DIFFUSION_INPUT.group_count + 1),
    yticks=np.arange(len(SPH_REFERENCE.region_labels)),
    yticklabels=SPH_REFERENCE.region_labels,
    xlabel='Energy group (fast to thermal)',
    title='CE normalized axial region/group flux reference',
)
fig.colorbar(image, ax=ax, label='normalized flux')
plt.tight_layout()
plt.show()

if power_mesh is not None:
    fig, ax = plt.subplots(figsize=(7.2, 6.0))
    image = ax.pcolormesh(
        power_mesh.r_edges_cm,
        power_mesh.z_edges_cm,
        power_mesh.mean.T,
        shading='flat',
        cmap='inferno',
    )
    for _, z_min, z_max in AXIAL_SPH_FUEL_ZONES:
        ax.axhline(z_min, color='white', lw=0.6, alpha=0.7)
        ax.axhline(z_max, color='white', lw=0.6, alpha=0.7)
    fig.colorbar(image, ax=ax, label='raw CE kappa-fission tally mean')
    ax.set(xlabel='r [cm]', ylabel='z [cm]', title='CE power mesh with axial SPH zone cuts')
    plt.tight_layout()
    plt.show()

In [ ]:
# call sph factors fit
SPH_RESULT = None
SPH_FAILURE = None
try:
    SPH_RESULT = fit_sph_factors(
        DIFFUSION_INPUT,
        spacing=MESH_SPACING,
        settings=SPH_SETTINGS,
        reference_mode=REFERENCE_MODE_AXIAL_REGION_FLUX,
        axial_sph_zones=AXIAL_SPH_ZONES,
    )
except SphConvergenceError as error:
    SPH_RESULT = error.result
    SPH_FAILURE = str(error)

print("=== AXIAL CE FLUX SPH FIXED-POINT RESULT ===")
print(f"Converged                  = {SPH_RESULT.factors.converged}")
print(f"Iterations                 = {SPH_RESULT.factors.iterations}")
print(f"Reference mode             = {SPH_RESULT.factors.reference_mode}")
print(f"Last k_eff                 = {SPH_RESULT.solution['k_eff']:.6f}")
print(f"Last CE delta-k            = {(SPH_RESULT.solution['k_eff'] - openmc_reference['keff']) * 1.0e5:+.1f} pcm")
print(f"Qualified                  = {SPH_RESULT.qualification['qualified']}")
print(f"Provisional reference      = {SPH_RESULT.qualification['provisional']}")
if SPH_FAILURE is not None:
    print(f"Status                     = {SPH_FAILURE}")
    print("The last iterate is diagnostic only; it is not a converged SPH correction.")
pprint(SPH_RESULT.qualification, sort_dicts=False)

# save results
SPH_FACTOR_PATH = MGXS_EXPORT_DIR / 'outputs' / 'sph_factors.json'
if SPH_RESULT.factors.converged and SPH_RESULT.qualification['qualified']:
    saved_path = save_sph_factors(SPH_RESULT.factors, SPH_FACTOR_PATH)
    SPH_PREPARED = prepare_concentric_diffusion_cache(
        DIFFUSION_INPUT,
        settings=CACHE_SETTINGS,
        sph_factors=SPH_RESULT.factors,
    )
    print(f"Qualified factors saved   = {saved_path}")
    print(f"Corrected cache directory = {SPH_PREPARED.cache_dir}")
else:
    print("No SPH artifact or corrected production cache was written.")
    print("Persistence requires convergence and all qualification thresholds.")

## 4. Exploratory Equivalent Rod Scan

This calculation adds a scalar absorption increment inside the physical rod radius. It does not use rodded OpenMC MGXS, SPH factors, or a transport calibration, and is excluded from the clean-core acceptance criteria.

In [ ]:
scan = scan_multigroup_rod_worth_2d(
    model_2d,
    x_values=np.linspace(0.0, 1.0, 5),
    mesh=mesh,
    clean_solution=sol_clean,
    max_iter=CACHE_SETTINGS.max_iter,
    tol=CACHE_SETTINGS.tol,
    source_tol=CACHE_SETTINGS.source_tol,
    max_inner_iter=CACHE_SETTINGS.max_inner_iter,
    inner_tol=CACHE_SETTINGS.inner_tol,
    warm_start=True,
)
if not np.isclose(scan['k_eff'][0], k_diff, rtol=0.0, atol=5.0e-5):
    raise RuntimeError('Rod scan zero-insertion state differs from the clean solve')

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(scan['x_insert'] * 100, scan['delta_rho_pcm'], 'o-', lw=2, label='equivalent rod worth')
ax.plot(scan['x_insert'] * 100, scan['rho_total_pcm'], 's--', lw=1.5, label='total reactivity')
ax.axhline(0.0, color='black', lw=0.7)
ax.set(xlabel='Rod insertion [%]', ylabel='Reactivity [pcm]', title='Exploratory uncalibrated rod scan')
ax.legend()
plt.tight_layout()
plt.show()

print("=== EXPLORATORY ROD SCAN ===")
print("Not transport-calibrated; excluded from validation.")
print(f"Full-insertion equivalent worth = {scan['delta_rho_pcm'][-1]:+.1f} pcm")
print(f"Full-insertion total rho        = {scan['rho_total_pcm'][-1]:+.1f} pcm")